# NB_SecurityDetails_Gold  
**Source:** `AI_Medallion.Silver.security`  
**Target:** `AI_Medallion.Gold.dim_security`  
**Load type:** Overwrite  
**Rule:** Direct. Select 5 columns from Silver and rename them (remove the `security_` prefix): email, report, access, division, access_scope.

In [1]:
# Workspace folder: C:\\spark-dev\\CoE_transformation_framework (see README)
import os
from pathlib import Path


os.environ["HADOOP_HOME"] = r"C:\hadoop\hadoop-3.5.0"
os.environ["HADOOP_PREFIX"] = os.environ["HADOOP_HOME"]
os.environ["PATH"] = os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep + os.environ["PATH"]
 
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("HADOOP_PREFIX:", os.environ["HADOOP_PREFIX"])
print("winutils exists:", os.path.isfile(
    os.path.join(os.environ["HADOOP_HOME"], "bin", "winutils.exe")
))

print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("HADOOP_PREFIX:", os.environ["HADOOP_PREFIX"])
print("winutils exists:", os.path.isfile(
    os.path.join(os.environ["HADOOP_HOME"], "bin", "winutils.exe")
))

HADOOP_HOME: C:\hadoop\hadoop-3.5.0
HADOOP_PREFIX: C:\hadoop\hadoop-3.5.0
winutils exists: True
HADOOP_HOME: C:\hadoop\hadoop-3.5.0
HADOOP_PREFIX: C:\hadoop\hadoop-3.5.0
winutils exists: True


In [2]:
_cwd = Path.cwd()
if not (_cwd / "Common" / "bootstrap.py").is_file():
    _repo = _cwd.parent
    if (_repo / "Common" / "bootstrap.py").is_file():
        os.chdir(_repo)

In [3]:
REBUILD_MEDALLION_CACHE = True  # True: rebuild; False: use cache
os.environ.setdefault("MEDALLION_DEBUG_READ", "1")
from Common.io import sync_rebuild_medallion_cache_env

sync_rebuild_medallion_cache_env(REBUILD_MEDALLION_CACHE)

In [4]:
import sys
print(sys.executable)

c:\spark-dev\CoE_transformation_framework\.venv\Scripts\python.exe


In [5]:
%run ./Common/bootstrap

Bootstrap OK: runtime=local mode=created spark=3.5.4 pyspark=3.5.4 path_reads=True onelake_read=deltalake-scan
Repo root: C:\spark-dev\CoE_Transformation_Framework_copilot
spark.jars.packages=io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-azure:3.3.4,com.azure:azure-storage-blob:12.25.1
OneLake reads: deltalake scan() + az login -> Spark (column mapping; avoids local PySpark abfss hang).
TABLE_CURSOR_BRONZE -> abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Bronze/cursor_usage
Tip: set MEDALLION_DEBUG_READ=1 before bootstrap to log read progress.
Tip: large tables — read_table() one at a time; transient Azure errors auto-retry.
Local table cache ON (default): parquet under ~/.fabric/medallion_cache/. Set REBUILD_MEDALLION_CACHE = True in settings to force OneLake. reload_io_helpers(globals()) after git pull — no kernel restart.
OneLake writes: deltalake + az login (merge/overwrite). Set MEDALLION_ONELAK

In [6]:
silver = read_table_broken_lineage(TABLE_SECURITY_SILVER, "security_silver")
silver.printSchema()

read_table_broken_lineage: scan -> local cache (AI_Medallion.Silver.security)
read_table: deltalake scan abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1e… (pass 1/6)
read_table: OneLake object-store concurrency limit=1 (raise MEDALLION_OBJECT_STORE_CONCURRENCY if reads are too slow)
read_table: deltalake scan() streaming batches…
read_table: scan streaming… 37 rows from OneLake so far
read_table: scan finished (295 rows)
read_table: opening local parquet cache (295 rows) -> C:\Users\soukelkr\.fabric\medallion_cache\AI_Medallion__Silver__security
read_table_broken_lineage: opened cache after scan (security_silver)
root
 |-- security_email: string (nullable = true)
 |-- security_report: string (nullable = true)
 |-- security_access: string (nullable = true)
 |-- security_codigo: string (nullable = true)
 |-- security_conocido: string (nullable = true)
 |-- security_nombre_completo: string (nullable = true)
 |-- security_pais: string (nullable = true)
 |-- 

In [7]:
gold = silver.select(
    F.col("security_email").alias("email"),
    F.col("security_report").alias("report"),
    F.col("security_access").alias("access"),
    F.col("security_division").alias("division"),
    F.col("security_access_scope").alias("access_scope"),
)
# Direct / Rename only: no filter, no dropDuplicates (Gold Scope: Processing Logic = Direct).

In [8]:
gold.printSchema()
gold.show(10, truncate=False)
print("silver rows:", silver.count(), "| gold rows:", gold.count())

root
 |-- email: string (nullable = true)
 |-- report: string (nullable = true)
 |-- access: string (nullable = true)
 |-- division: string (nullable = true)
 |-- access_scope: string (nullable = true)

+------------------------------+------+------+--------+------------+
|email                         |report|access|division|access_scope|
+------------------------------+------+------+--------+------------+
|jferngut@emeal.nttdata.com    |INC   |Yes   |NULL    |ALL         |
|shaakir.khan@dimensiondata.com|Tokens|Yes   |MEA     |DIVISION    |
|jtorrbec@emeal.nttdata.com    |INC   |Yes   |IBIOL   |DIVISION    |
|eroura@emeal.nttdata.com      |Tokens|Yes   |IBIOL   |DIVISION    |
|eperezmm@emeal.nttdata.com    |Tokens|Yes   |IBIOL   |DIVISION    |
|Laura.Loeer@bs.nttdata.com    |INC   |Yes   |BS      |DIVISION    |
|matias.gonzalez@nttdata.com   |INC   |Yes   |IBIOL   |DIVISION    |
|Sgarodri@emeal.nttdata.com    |INC   |Yes   |IBIOL   |DIVISION    |
|jose12.herrera@global.ntt     |INC   

In [9]:
apply_local_merge_spark_conf(spark)

write_full_table(gold, TABLE_DIM_SECURITY)

show_sample(gold.limit(10))

publish_merge_staging: AI_Medallion.Gold.dim_security -> C:\Users\soukelkr\.fabric\merge_staging\AI_Medallion__Gold__dim_security
materialize staging: Spark JVM parquet (8 partitions)
deltalake write mode=overwrite abfss://4cecee03-0629-4081-8ae8-b749ee04d43f@onelake.dfs.fabric.microsoft.com/1ee46463-2495-44f5-9dc9-d635d0067483/Tables/Gold/dim_security (local staging -> OneLake)...
deltalake write chunk 1 (37 rows) mode=overwrite
deltalake write chunk 2 (36 rows) mode=append
deltalake write chunk 3 (37 rows) mode=append
deltalake write chunk 4 (37 rows) mode=append
deltalake write chunk 5 (37 rows) mode=append
deltalake write chunk 6 (37 rows) mode=append
deltalake write chunk 7 (37 rows) mode=append
deltalake write chunk 8 (37 rows) mode=append
deltalake write done


DataFrame[email: string, report: string, access: string, division: string, access_scope: string]